In [1]:
# ==========================================
# CELDA 1: INSTALAR LIBRERÍAS
# ==========================================

!pip install -q \
  langchain \
  langchain-community \
  langchain-experimental \
  langchain-mistralai \
  faiss-cpu \
  tabulate \
  pandas==2.2.2 \
  requests==2.32.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/461.3 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 22.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph 1.1.9 requires langchain-core<2,>=1.3.0, but you have langchain-core 0.3.86 which is incompatible.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.3.86 which is incompatible.


In [2]:
# ==========================================
# CELDA 2: IMPORTAR LIBRERÍAS
# ==========================================

import os
import io
import json
import pandas as pd
import numpy as np

from google.colab import files, userdata
from sklearn.preprocessing import MinMaxScaler

from langchain_core.documents import Document
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_experimental.agents import create_pandas_dataframe_agent

In [3]:
# ==========================================
# CELDA 3: SUBIR Y CARGAR DATASET
# ==========================================

print("Subí tu archivo CSV, por ejemplo: sales_data_sample.csv")
uploaded = files.upload()

if not uploaded:
    raise ValueError("No se subió ningún archivo.")

filename = next(iter(uploaded.keys()))

try:
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
except UnicodeDecodeError:
    df = pd.read_csv(io.BytesIO(uploaded[filename]), encoding="latin1")

df.columns = df.columns.str.strip()

if "ORDERDATE" in df.columns:
    df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"], errors="coerce")

print(f"Dataset cargado: {filename}")
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
display(df.head())

Subí tu archivo CSV, por ejemplo: sales_data_sample.csv


Saving dataset_normalizado.csv to dataset_normalizado.csv
Dataset cargado: dataset_normalizado.csv
Filas: 2823 | Columnas: 25


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,0.021538,0.263736,0.941193,0.058824,0.175644,2003-02-24,Shipped,0.000000,0.090909,0.0,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,0.064615,0.307692,0.744940,0.235294,0.167916,2003-05-07,Shipped,0.333333,0.363636,0.0,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,0.104615,0.384615,0.928063,0.058824,0.250150,2003-07-01,Shipped,0.666667,0.545455,0.0,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,0.138462,0.428571,0.771061,0.294118,0.240030,2003-08-25,Shipped,0.666667,0.636364,0.0,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,0.181538,0.472527,1.000000,0.764706,0.347273,2003-10-10,Shipped,1.000000,0.818182,0.0,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [4]:
# ==========================================
# CELDA 4: INFORMACIÓN GENERAL
# ==========================================

print("Dimensiones del dataset:")
print(df.shape)

print("\nInformación general:")
print(df.info())

print("\nValores nulos por columna:")
print(df.isnull().sum())

Dimensiones del dataset:
(2823, 25)

Información general:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2823 entries, 0 to 2822
Data columns (total 25 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   ORDERNUMBER       2823 non-null   float64       
 1   QUANTITYORDERED   2823 non-null   float64       
 2   PRICEEACH         2823 non-null   float64       
 3   ORDERLINENUMBER   2823 non-null   float64       
 4   SALES             2823 non-null   float64       
 5   ORDERDATE         2823 non-null   datetime64[ns]
 6   STATUS            2823 non-null   object        
 7   QTR_ID            2823 non-null   float64       
 8   MONTH_ID          2823 non-null   float64       
 9   YEAR_ID           2823 non-null   float64       
 10  PRODUCTLINE       2823 non-null   object        
 11  MSRP              2823 non-null   float64       
 12  PRODUCTCODE       2823 non-null   object        
 13  CUSTOMERNAME      28

In [5]:
# ==========================================
# CELDA 5: LIMPIEZA BÁSICA
# ==========================================

df = df.drop_duplicates()

columnas_numericas = df.select_dtypes(include=["int64", "float64"]).columns

for columna in columnas_numericas:
    df[columna] = df[columna].fillna(df[columna].mean())

print("Duplicados eliminados y valores nulos numéricos corregidos.")

Duplicados eliminados y valores nulos numéricos corregidos.


In [6]:
# ==========================================
# CELDA 6: NORMALIZACIÓN MIN-MAX
# ==========================================

columnas_numericas = df.select_dtypes(include=["int64", "float64"]).columns

scaler = MinMaxScaler()
df[columnas_numericas] = scaler.fit_transform(df[columnas_numericas])

print("Dataset normalizado correctamente con MinMaxScaler.")
display(df.head())

Dataset normalizado correctamente con MinMaxScaler.


,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,0.021538,0.263736,0.941193,0.058824,0.175644,2003-02-24,Shipped,0.000000,0.090909,0.0,...,897 Long Airport Avenue,NaN,NYC,NY,10022,USA,NaN,Yu,Kwai,Small
1,0.064615,0.307692,0.744940,0.235294,0.167916,2003-05-07,Shipped,0.333333,0.363636,0.0,...,59 rue de l'Abbaye,NaN,Reims,NaN,51100,France,EMEA,Henriot,Paul,Small
2,0.104615,0.384615,0.928063,0.058824,0.250150,2003-07-01,Shipped,0.666667,0.545455,0.0,...,27 rue du Colonel Pierre Avia,NaN,Paris,NaN,75508,France,EMEA,Da Cunha,Daniel,Medium
3,0.138462,0.428571,0.771061,0.294118,0.240030,2003-08-25,Shipped,0.666667,0.636364,0.0,...,78934 Hillside Dr.,NaN,Pasadena,CA,90003,USA,NaN,Young,Julie,Medium
4,0.181538,0.472527,1.000000,0.764706,0.347273,2003-10-10,Shipped,1.000000,0.818182,0.0,...,7734 Strong St.,NaN,San Francisco,CA,NaN,USA,NaN,Brown,Julie,Medium


In [7]:
# ==========================================
# CELDA 7: GUARDAR DATASET NORMALIZADO
# ==========================================

nombre_archivo = "dataset_normalizado.csv"
df.to_csv(nombre_archivo, index=False)

print(f"Dataset guardado como: {nombre_archivo}")

Dataset guardado como: dataset_normalizado.csv


In [8]:
# ==========================================
# CELDA 8: CONFIGURAR API KEY DE MISTRAL
# ==========================================

api_key = userdata.get("MISTRAL_API_KEY")

if not api_key:
    raise ValueError(
        "No encontré MISTRAL_API_KEY en Colab Secrets. "
        "Creá el secret en el panel de la llave y volvé a ejecutar esta celda."
    )

os.environ["MISTRAL_API_KEY"] = api_key

print("API key configurada correctamente.")

API key configurada correctamente.


In [9]:
# ==========================================
# CELDA 9: CREAR DOCUMENTOS PARA RAG
# ==========================================

documentos = []

documentos.append(Document(
    page_content=f"""
Dataset de ventas normalizado.

Filas: {df.shape[0]}
Columnas: {df.shape[1]}
Columnas disponibles: {list(df.columns)}

Este dataset contiene información de órdenes, productos, clientes,
fechas, países, estados de orden y métricas comerciales.

Las columnas numéricas fueron normalizadas con MinMaxScaler.
Por eso, los valores numéricos están escalados normalmente entre 0 y 1.
No representan los valores originales.
""",
    metadata={"tipo": "descripcion_general"}
))

for col in df.columns:
    texto = f"""
Columna: {col}
Tipo de dato: {df[col].dtype}
Valores nulos: {df[col].isnull().sum()}
Ejemplos de valores: {df[col].dropna().head(10).tolist()}
"""

    if pd.api.types.is_numeric_dtype(df[col]):
        texto += f"""
Resumen numérico:
{df[col].describe().to_string()}
"""
    else:
        texto += f"""
Valores más frecuentes:
{df[col].value_counts(dropna=False).head(10).to_string()}
"""

    documentos.append(Document(
        page_content=texto,
        metadata={"tipo": "columna", "columna": col}
    ))

documentos.append(Document(
    page_content=f"""
Estadísticas numéricas generales:

{df.describe(include="number").round(4).to_string()}
""",
    metadata={"tipo": "estadisticas_generales"}
))

print(f"Documentos creados para RAG: {len(documentos)}")

Documentos creados para RAG: 27


In [10]:
# ==========================================
# CELDA 10: CREAR EMBEDDINGS + FAISS
# ==========================================

embeddings = MistralAIEmbeddings(
    model="mistral-embed"
)

vectorstore = FAISS.from_documents(
    documents=documentos,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

print("RAG creado correctamente con FAISS + embeddings de Mistral.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer.json: 0.00B [00:00, ?B/s]

RAG creado correctamente con FAISS + embeddings de Mistral.


In [11]:
# ==========================================
# CELDA 11: CREAR CONTEXTO BASE DEL AGENTE
# ==========================================

numeric_summary = df.describe(include="number").round(4).to_string()

categorical_cols = [
    "STATUS", "PRODUCTLINE", "DEALSIZE", "COUNTRY",
    "TERRITORY", "YEAR_ID", "QTR_ID", "MONTH_ID"
]

cat_summary = {
    col: df[col].value_counts(dropna=False).head(20).to_dict()
    for col in categorical_cols
    if col in df.columns
}

def valores_unicos(col):
    if col in df.columns:
        return df[col].dropna().unique().tolist()
    return "Columna no disponible"

DATASET_CONTEXT_RAW = f"""
Eres un agente experto en análisis de datos de ventas.
Trabajas con un DataFrame de pandas llamado `df`.

El dataset ya fue limpiado y normalizado con MinMaxScaler.
Por eso, las columnas numéricas están escaladas normalmente entre 0 y 1.
Cuando respondas sobre valores numéricos normalizados, aclara que no son
valores originales, sino valores escalados con Min-Max.

=== ESTRUCTURA ===
- Filas: {df.shape[0]}
- Columnas: {df.shape[1]}
- Columnas disponibles: {list(df.columns)}

=== COLUMNAS IMPORTANTES ===
- ORDERNUMBER: Identificador de la orden.
- QUANTITYORDERED: Cantidad pedida normalizada.
- PRICEEACH: Precio unitario normalizado.
- SALES: Venta normalizada.
- ORDERDATE: Fecha de la orden.
- STATUS: Estado de la orden. Valores: {valores_unicos("STATUS")}
- PRODUCTLINE: Línea de producto. Valores: {valores_unicos("PRODUCTLINE")}
- COUNTRY: País.
- TERRITORY: Territorio. Valores: {valores_unicos("TERRITORY")}
- DEALSIZE: Tamaño del trato. Valores: {valores_unicos("DEALSIZE")}

=== ESTADÍSTICAS NUMÉRICAS ===
{numeric_summary}

=== DISTRIBUCIÓN CATEGÓRICA ===
{json.dumps(cat_summary, indent=2, ensure_ascii=False)}

Instrucciones:
1. Usa siempre el DataFrame `df` para cálculos reales.
2. Responde siempre en español.
3. Si una columna no existe, dilo claramente.
4. Usa el contexto RAG solo como apoyo, pero calcula con pandas.
"""

DATASET_CONTEXT = DATASET_CONTEXT_RAW.replace("{", "{{").replace("}", "}}")

print("Contexto base del agente construido correctamente.")

Contexto base del agente construido correctamente.


In [12]:
# ==========================================
# CELDA 12: CREAR MODELO Y AGENTE PANDAS
# ==========================================

llm = ChatMistralAI(
    model="mistral-small-latest",
    temperature=0,
)

try:
    agent = create_pandas_dataframe_agent(
        llm=llm,
        df=df,
        agent_type="tool-calling",
        verbose=True,
        allow_dangerous_code=True,
        prefix=DATASET_CONTEXT,
        number_of_head_rows=5,
        agent_executor_kwargs={"handle_parsing_errors": True},
    )
except Exception as error_tool_calling:
    print("No se pudo iniciar con tool-calling. Se usará zero-shot-react-description.")
    print(f"Detalle: {error_tool_calling}")

    agent = create_pandas_dataframe_agent(
        llm=llm,
        df=df,
        agent_type="zero-shot-react-description",
        verbose=True,
        allow_dangerous_code=True,
        prefix=DATASET_CONTEXT,
        number_of_head_rows=5,
        agent_executor_kwargs={"handle_parsing_errors": True},
    )

print("Agente Pandas con Mistral listo.")

Agente Pandas con Mistral listo.


In [13]:
# ==========================================
# CELDA 13: FUNCIÓN PARA PREGUNTAR CON RAG
# ==========================================

def preguntar_rag(pregunta: str, mostrar_contexto: bool = False):
    if not pregunta or not pregunta.strip():
        raise ValueError("La pregunta no puede estar vacía.")

    docs_relevantes = retriever.invoke(pregunta)

    contexto_rag = "\n\n".join(
        [doc.page_content for doc in docs_relevantes]
    )

    pregunta_aumentada = f"""
Usa el siguiente contexto recuperado mediante RAG para orientar tu análisis.

=== CONTEXTO RAG ===
{contexto_rag}

=== PREGUNTA DEL USUARIO ===
{pregunta}

Instrucciones:
- Usa el DataFrame `df` para hacer cálculos reales.
- Responde en español.
- Sé claro y directo.
- Si usas columnas numéricas, aclara que están normalizadas con MinMaxScaler.
- No inventes columnas que no existan.
"""

    if mostrar_contexto:
        print("Contexto recuperado por RAG:")
        print(contexto_rag)
        print("\n" + "=" * 70 + "\n")

    respuesta = agent.invoke({"input": pregunta_aumentada})
    salida = respuesta.get("output", respuesta)

    print("Pregunta:")
    print(pregunta)

    print("\nRespuesta:")
    print(salida)

    return salida

In [14]:
# ==========================================
# CELDA 14: EJEMPLOS DE USO
# ==========================================

preguntar_rag("¿Cuántas órdenes hay por cada línea de producto?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "df['PRODUCTLINE'].value_counts()"}`


PRODUCTLINE
Classic Cars        967
Vintage Cars        607
Motorcycles         331
Planes              306
Trucks and Buses    301
Ships               234
Trains               77
Name: count, dtype: int64Hay **2823 órdenes** distribuidas en las siguientes líneas de producto:

- **Classic Cars**: 967 órdenes
- **Vintage Cars**: 607 órdenes
- **Motorcycles**: 331 órdenes
- **Planes**: 306 órdenes
- **Trucks and Buses**: 301 órdenes
- **Ships**: 234 órdenes
- **Trains**: 77 órdenes

> Finished chain.
Pregunta:
¿Cuántas órdenes hay por cada línea de producto?

Respuesta:
Hay **2823 órdenes** distribuidas en las siguientes líneas de producto:

- **Classic Cars**: 967 órdenes
- **Vintage Cars**: 607 órdenes
- **Motorcycles**: 331 órdenes
- **Planes**: 306 órdenes
- **Trucks and Buses**: 301 órdenes
- **Ships**: 234 órdenes
- **Trains**: 77 órdenes


'Hay **2823 órdenes** distribuidas en las siguientes líneas de producto:\n\n- **Classic Cars**: 967 órdenes\n- **Vintage Cars**: 607 órdenes\n- **Motorcycles**: 331 órdenes\n- **Planes**: 306 órdenes\n- **Trucks and Buses**: 301 órdenes\n- **Ships**: 234 órdenes\n- **Trains**: 77 órdenes'

In [15]:
# ==========================================
# CELDA 15: MÁS CONSULTAS
# Ejecuta una por vez para evitar rate limit
# ==========================================

preguntar_rag("¿Qué país tiene más órdenes registradas?")



> Entering new AgentExecutor chain...

Invoking: `python_repl_ast` with `{'query': "# Contar el número de órdenes por país\nordenes_por_pais = df['COUNTRY'].value_counts()\n\n# Obtener el país con más órdenes\npais_mas_ordenes = ordenes_por_pais.idxmax()\ncantidad_ordenes = ordenes_por_pais.max()\n\npais_mas_ordenes, cantidad_ordenes"}`


('USA', 1004)El país con más órdenes registradas es **USA**, con un total de **1004 órdenes**.

> Finished chain.
Pregunta:
¿Qué país tiene más órdenes registradas?

Respuesta:
El país con más órdenes registradas es **USA**, con un total de **1004 órdenes**.


'El país con más órdenes registradas es **USA**, con un total de **1004 órdenes**.'

In [16]:
# ==========================================
# CELDA 16: CONSULTA EJECUTIVA
# ==========================================

preguntar_rag("Dame un resumen ejecutivo del dataset y menciona los datos más importantes.")



> Entering new AgentExecutor chain...
### **Resumen Ejecutivo del Dataset de Ventas**

#### **1. Descripción General**
El dataset contiene **2,823 registros** de órdenes de ventas con **25 columnas** que incluyen información sobre productos, clientes, fechas, ubicaciones y métricas comerciales. **Todas las columnas numéricas están normalizadas con MinMaxScaler**, por lo que sus valores están escalados entre **0 y 1** (no representan los valores originales).

---

#### **2. Métricas Clave (Valores Normalizados)**
| Métrica               | Valor Promedio | Valor Máximo | Valor Mínimo | Desviación Estándar |
|-----------------------|----------------|--------------|--------------|---------------------|
| **SALES** (Ventas)    | 0.2259         | 1.0000       | 0.0000       | 0.1354              |
| **ORDERNUMBER**       | 0.4884         | 1.0000       | 0.0000       | 0.2833              |
| **QUANTITYORDERED**   | 0.3197         | 1.0000       | 0.0000       | 0.1070              |
| **P

'### **Resumen Ejecutivo del Dataset de Ventas**\n\n#### **1. Descripción General**\nEl dataset contiene **2,823 registros** de órdenes de ventas con **25 columnas** que incluyen información sobre productos, clientes, fechas, ubicaciones y métricas comerciales. **Todas las columnas numéricas están normalizadas con MinMaxScaler**, por lo que sus valores están escalados entre **0 y 1** (no representan los valores originales).\n\n---\n\n#### **2. Métricas Clave (Valores Normalizados)**\n| Métrica               | Valor Promedio | Valor Máximo | Valor Mínimo | Desviación Estándar |\n|-----------------------|----------------|--------------|--------------|---------------------|\n| **SALES** (Ventas)    | 0.2259         | 1.0000       | 0.0000       | 0.1354              |\n| **ORDERNUMBER**       | 0.4884         | 1.0000       | 0.0000       | 0.2833              |\n| **QUANTITYORDERED**   | 0.3197         | 1.0000       | 0.0000       | 0.1070              |\n| **PRICEEACH**         | 0.776